# Milestone 3 — Train GCN on Cora, audit against bracket — **E04 in miniature**

This milestone reproduces **E04** ([`PAPER-ARXIV.md`](../../../../PAPER-ARXIV.md) §7.1, Trained-GNN Correspondence) in miniature: 1 dataset × 1 architecture × 1 seed, vs. the full E04 sweep of 3 datasets × 4 archs × 5 seeds. Trains a 2-layer GCN on Cora **on a real Modal T4 GPU** (you'll watch the cell stream logs back) and audits the empirical error against the M2 bracket. A CPU mirror runs first for sanity and CI.

**Modal-in-notebook crash course.** Modal apps are Python objects with `@app.function(gpu=...)` decorators. Inside a notebook you (1) import the `app` and decorated functions, (2) open a context with `app.run()` (allocates the container), (3) call `fn.remote(...)` — args pickled, work runs on the GPU, result returns as a normal Python object. First call cold-starts ~30s; subsequent calls reuse the warm container.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='m3', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from onboarding.projects.shared import (
    Partition, label_partition, wl_partition,
    hbin, hbin_inverse, upper, lower, slack, verify, bracket_of,
    sum_partition, mean_partition, max_partition,
)
print('shared modules imported')


In [ ]:
from torch_geometric.datasets import Planetoid
_CACHE = Path.home() / '.cache' / 'planetoid'
_CACHE.mkdir(parents=True, exist_ok=True)
_cora = Planetoid(root=str(_CACHE / 'Cora'), name='Cora')[0]
n = int(_cora.num_nodes)
m = int(_cora.num_edges)
K = int(_cora.y.max().item()) + 1
print(f'Cora: n={n}, m={m}, classes={K}, x.shape={tuple(_cora.x.shape)}')


In [ ]:
import torch, torch.nn.functional as F
from torch_geometric.nn import GCNConv
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cpu')

class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.c1 = GCNConv(in_dim, hidden)
        self.c2 = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        x = F.relu(self.c1(x, edge_index))
        return self.c2(x, edge_index)

data = _cora.to(device)
model = GCN(data.num_features, 64, K).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(200):
    model.train(); opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); opt.step()
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=-1)
    test_acc = float((pred[data.test_mask] == data.y[data.test_mask]).float().mean())
    full_err = float((pred != data.y).float().mean())
print(f'GCN test acc = {test_acc:.4f}  | full-graph error = {full_err:.4f}')


**Audit against the bracket** — the empirical error of any classifier must respect $\mathrm{lower}(H) \le \varepsilon^{\text{model}}$ when restricted to a coarsening; we additionally check the relation to the WL(L=2) bracket.

In [ ]:
P = wl_partition(_cora, depth=2)
bin_e = np.array([min(e, 1-e) for e in P.e])
q = np.array(P.q)
H = float(np.sum(q * np.array([hbin(e) for e in bin_e])))
print(f'wl(L=2): H={H:.4f}, lower={lower(H):.4f}, upper={upper(H):.4f}, GCN err={full_err:.4f}')


In [ ]:
assert test_acc > 0.70, f'M3: GCN test acc {test_acc} below 0.70'
# Standard sanity: GCN full-graph error should not exceed the upper bracket of wl(L=2) by a wide margin.
assert full_err <= upper(H) + 0.20, f'M3: GCN err {full_err} >> upper(H)={upper(H)} — suspicious'
print(f'[GATE OK] M3: GCN test acc {test_acc:.3f}; full err {full_err:.3f} within plausible window of upper(H)={upper(H):.3f}')


In [ ]:
reflect.log('M3.gcn_audit', f'GCN test acc={test_acc:.3f}; bracket upper(H)={upper(H):.3f} on wl(L=2)', 'HIGH')


### Modal GPU run — the actual T4 training

This cell **will** spin up a T4 on Modal, train the GCN there, and stream the result back. Requires `pip install modal && modal token new` to have been run once on this machine.

In [ ]:
from onboarding.projects import modal_app
if not (modal_app._MODAL_OK and modal_app.app is not None):
    raise RuntimeError('Modal not installed/configured. Run `pip install modal && modal token new`, then re-run this cell.')
print('opening Modal app...', flush=True)
with modal_app.app.run():
    print('ping:', modal_app.ping.remote(), flush=True)
    gpu_result = modal_app.train_gcn.remote(epochs=200)
print('GPU training result:', gpu_result)


**Gate M3-GPU.** The remote container ran on `cuda` and matched or exceeded the CPU test accuracy.

In [ ]:
assert gpu_result['device'] == 'cuda', f'M3-GPU: expected cuda device, got {gpu_result["device"]}'
assert gpu_result['test_acc'] > 0.70, f'M3-GPU: GPU test acc {gpu_result["test_acc"]} below 0.70'
assert abs(gpu_result['test_acc'] - test_acc) < 0.10, f'M3-GPU: GPU vs CPU accuracy diverged: gpu={gpu_result["test_acc"]:.3f}, cpu={test_acc:.3f}'
print(f'[GATE OK] M3-GPU: T4 ran {gpu_result["epochs"]} epochs; test_acc={gpu_result["test_acc"]:.3f}; CPU mirror was {test_acc:.3f}')


In [ ]:
reflect.log('M3.modal_gpu', f'T4 trained GCN reached test_acc={gpu_result["test_acc"]:.3f} on Cora', 'HIGH')


**End of M3.**